# Práctica autónoma · Observatorio meteorológico
Sin nota. Tiempo sugerido: 90–150 minutos; S6–S8 son ampliación. Entrega personal sugerida: código, resultados y explicaciones. No contiene soluciones del trabajo final.

Lee primero la magistral. Para SQL usa la base de práctica y la carga inicial de la guía; psql se ejecuta en Ubuntu, no en una celda Python. El CSV Pandas tiene ambos meses: filtra enero cuando compares con Silver antes de actualizar. No se instala infraestructura adicional.


In [ ]:
from pathlib import Path
from io import BytesIO
import os, sys, hashlib, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
pd.set_option('display.max_columns', 12)
print('Python:', sys.version.split()[0], '| Pandas:', pd.__version__)


In [ ]:
URL_CSV = 'https://raw.githubusercontent.com/Santiago-Echeverri-Arteaga/Programacion/master/notebooks/cierre_pandas_sql_2026/datos/meteorologia.csv'
RAIZ = next((r for r in [Path.cwd(), Path.cwd()/'notebooks/cierre_pandas_sql_2026']
             if (r/'datos/meteorologia.csv').exists()), None)
fuente = RAIZ/'datos/meteorologia.csv' if RAIZ else (Path('meteorologia.csv') if Path('meteorologia.csv').exists() else URL_CSV)
datos_originales = pd.read_csv(fuente, parse_dates=['fecha'])
print('Filas y columnas:', datos_originales.shape)
display(datos_originales.head())
# Internet: pd.read_csv(URL_CSV, parse_dates=['fecha'])
# PC: pd.read_csv(r'C:/ruta/meteorologia.csv', parse_dates=['fecha'])


In [ ]:
def preparar_tabla(original):
    tabla = original.copy()
    tabla['ciudad_id'] = tabla['ciudad_id'].astype('string').str.strip().str.lower()
    tabla['fecha'] = pd.to_datetime(tabla['fecha'], errors='raise')
    tabla['mes'] = tabla['fecha'].dt.to_period('M').astype(str)
    tabla['tmax_k'] = tabla['tmax_c'] + 273.15
    tabla['amplitud_c'] = tabla['tmax_c'] - tabla['tmin_c']
    return tabla

datos = preparar_tabla(datos_originales)


## P1 · Selección reproducible
Selecciona Barranquilla con máxima de al menos 30 °C. Conserva ciudad, fecha y máxima; ordena por temperatura descendente y fecha ascendente. Explica el desempate y compara > con >=.


In [ ]:
# Escribe tu solución y añade una celda Markdown con la interpretación.


## P2 · Ausencias
Cuenta ausencias de máxima, mínima y precipitación por separado. ¿Cuántas filas perderías al exigir las tres? Si no hay ausencias, informa ese resultado sin inventarlas. Explica por qué una ausencia futura de precipitación no obliga a excluir temperatura y por qué cero mm tiene otro significado.


In [ ]:
# Escribe tu solución y añade una celda Markdown con la interpretación.


## P3 · Unidades y copia
Crea una copia con máximas en kelvin sin cambiar datos_originales. Verifica a mano tres conversiones y su promedio. ¿Cambian el conteo de presentes, la desviación estándar o las diferencias de temperatura al sumar 273.15? Justifica.


In [ ]:
# Escribe tu solución y añade una celda Markdown con la interpretación.


## P4 · Resumen por ciudad
Calcula filas, máximas presentes, media y desviación estándar por ciudad, con cobertura porcentual. Contrasta cobertura respecto de filas recibidas y respecto del calendario. Grafica con unidades y explica por qué la comparación no aísla el efecto de altitud.


In [ ]:
# Escribe tu solución y añade una celda Markdown con la interpretación.


## P5 · Unión controlada
Une la media de cada ciudad a cada día; conserva filas y calcula desviaciones respecto de esa media. Explica qué validación detectaría una ciudad repetida en la tabla de medias. Compara con transform y aclara por qué no obtienes una anomalía climatológica.


In [ ]:
# Escribe tu solución y añade una celda Markdown con la interpretación.


## P6 · Forma y exportación
Construye medias de máximas por ciudad (filas) y mes (columnas); vuelve a formato largo. Compara pivot con pivot_table. Explica qué información se pierde. Reúne meses con concat y revisa claves. Exporta CSV, comprueba su lectura y registra procedencia; Drive es opcional.


In [ ]:
# Escribe tu solución y añade una celda Markdown con la interpretación.


## S1 · Filtrar y ordenar
Obtén los cinco días de Armenia con mayor precipitación; muestra fecha, precipitación y máxima. Excluye ausentes y resuelve empates por fecha. Reproduce en Pandas para el mismo periodo y compara.

```sql
-- Escribe las consultas necesarias en psql o en un archivo de práctica.
```

Anota interpretación y comprobaciones; no basta con mostrar la salida.


## S2 · NULL, claves y conteos
Por ciudad informa filas, máximas presentes y ausentes mediante dos conteos. Explica por qué tmax_c = NULL no funciona. Justifica la clave primaria compuesta: ¿por qué no basta ciudad o fecha? Consulta claves repetidas y explica qué esperas antes de ejecutar.

```sql
-- Escribe las consultas necesarias en psql o en un archivo de práctica.
```

Anota interpretación y comprobaciones; no basta con mostrar la salida.


## S3 · WHERE y HAVING
Resume por ciudad entre máximas presentes y conserva grupos con media superior a 24 °C. Explica la diferencia con filtrar días superiores a 24 antes de agrupar. Escribe una pregunta en lenguaje natural para cada consulta.

```sql
-- Escribe las consultas necesarias en psql o en un archivo de práctica.
```

Anota interpretación y comprobaciones; no basta con mostrar la salida.


## S4 · Conservar categorías
Cuenta días de precipitación >=50 mm por ciudad, incluidas ciudades con cero casos. Decide INNER o LEFT JOIN, dónde poner el filtro y qué columna contar. Explica cómo una unión de dos tablas diarias solo por ciudad multiplicaría resultados.

```sql
-- Escribe las consultas necesarias en psql o en un archivo de práctica.
```

Anota interpretación y comprobaciones; no basta con mostrar la salida.


## S5 · Transacción y restricciones
En una transacción crea una tabla temporal ensayo_practica con clave primaria y una medida no negativa, inserta una fila artificial válida, consulta y revierte. Explica qué rechaza cada restricción, cómo trata NULL un CHECK y por qué la tabla desaparece tras ROLLBACK. No cambies los datos reales.

```sql
-- Escribe las consultas necesarias en psql o en un archivo de práctica.
```

Anota interpretación y comprobaciones; no basta con mostrar la salida.


## S6 · Pipeline y actualización
En una base de pruebas nueva ejecuta inicial y consulta Gold. Predice conteos, ejecuta actualización y vuelve a consultar Gold. Repite actualización y comprueba que se conservan filas previas sin duplicados. Explica por qué DO NOTHING no corrige valores ya guardados. Si ya actualizaste tu base, pide una base nueva; no borres datos para reiniciarla.

```sql
-- Escribe las consultas necesarias en psql o en un archivo de práctica.
```

Anota interpretación y comprobaciones; no basta con mostrar la salida.


## S7 · Cambiar una consulta
En una copia de sql/resumen_mensual.sql agrega un filtro para devolver solo una ciudad. Usa esa copia en una copia del pipeline y adapta el informe si cambias columnas. Verifica qué recibe Python y cuándo aparece COMPLETADO.txt. Ensaya un error de consulta en la copia: ¿hay carpeta parcial?, ¿qué cambios de base permanecen?

```sql
-- Escribe las consultas necesarias en psql o en un archivo de práctica.
```

Anota interpretación y comprobaciones; no basta con mostrar la salida.


## S8 · Localizar cálculo y memoria
Dibuja PostgreSQL remoto de 500 GB y Python local. Compara SELECT *, WHERE y GROUP BY; marca qué cruza la red y qué queda almacenado. Explica por qué 100 filas de resumen no necesitan 500 GB de RAM. Propón una operación por lotes y otra que necesite estado entre lotes. Distingue Python remoto de procesamiento distribuido.

```sql
-- Escribe las consultas necesarias en psql o en un archivo de práctica.
```

Anota interpretación y comprobaciones; no basta con mostrar la salida.
